# VATEX - część deweloperska

Przygotowuje polecenia pozyskania części deweloperskiej VATEX, kontroluje ich wynik i składa plik zapytań. Sam niczego nie pobiera. Korzysta wyłącznie z biblioteki standardowej, więc działa niezależnie od stanu środowiska z PyTorch i może chodzić równolegle z ekstrakcją cech.

**Do czego służy część deweloperska.** Wyłącznie do wyznaczenia progu dopasowania fraz do nazw klas i do kontroli sygnału ruchu. Nie dostaje znaczników wymagań ani etykiety złożoności i nie jest podstawą żadnej innej decyzji.

**Dlaczego ze splitu treningowego, a nie z wycięcia kawałka części testowej.** Część testowa liczy 2489 klipów i tyle samo wynosi liczba fragmentów 18 odcinków serialu; ta równość jest warunkiem porównywania wyników między domenami i ma zostać nienaruszona. Wydzielenie części deweloperskiej z testowej odebrałoby jej tę własność, więc klipy deweloperskie pochodzą z osobnej puli: ze splitu treningowego VATEX, rozłącznego z walidacyjnym, ale zbudowanego na tej samej podstawie (zbiór walidacyjny Kinetics-600).

**Wymaga:** plików `vatex_training_v1.0.json`, `kinetics-600_val.csv` i `kinetics-400_train.csv` w `data/interim/vatex` oraz skryptów z tabeli w sekcji niżej.

**Zapisuje:**

| plik | co zawiera | gdzie powstaje |
|---|---|---|
| `data/interim/vatex/vatex_dev_candidates.txt` | zamrożona lista wylosowanych identyfikatorów | `vatex_candidates.py` |
| `data/interim/vatex/work/vatex_descriptions_dev.csv` | opisy klipów, eksport z jsona | `vatex_download.py --split dev --mode download` |
| `data/interim/vatex/vatex_report_dev.csv` | rejestr pobrania: status, rozdzielczość, fps, długość | `vatex_download.py --split dev --mode download` |
| `data/interim/vatex/vatex_split_dev.csv` | przeciek K400 i podział klas | `vatex_leak_filter.py --split dev` |
| `data/interim/vatex/work/clips_too_short_dev.csv` | klipy krótsze niż okno adnotacji, wykluczone automatycznie | blok 4 |
| `data/annotations/vatex/vatex_queries_dev.jsonl` | zapytania części deweloperskiej, jeden opis na klip | blok 5 |

Lista kandydatów jest zamrożona: powstaje raz, przed pobieraniem, i nie jest przelosowywana - `vatex_candidates.py` bez `--force` odmawia jej nadpisania. Co nie pobierze się z YouTube, przepada i nic nie wchodzi w jego miejsce. Rejestr pobrania jest dopisywany, stąd wznawialność; pozostałe pliki są nadpisywane.

**Dalej:** `vatex_02_test_acquisition.ipynb` - pozyskanie części testowej.

In [ ]:
# Standard library only, on purpose: this notebook has to run while the PyTorch
# environment is busy extracting features. src.utils.queries imports nothing
# else either; src.utils.vatex does (pandas), so it is NOT imported here.

import csv
import hashlib
import json
import sys
from collections import Counter
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import settings
from src.utils.queries import Query, save_jsonl

CONFIG = {
    "split": "dev",          # part of the dataset prepared here
    "sample_size": 750,      # clips drawn by vatex_candidates.py
    "seed": 1234,            # draw seed, frozen together with the list
    # which enCap description becomes the query -- taken from settings, not
    # repeated here: the test part reads the same constant, and a rule that
    # differed between the parts would make the two files incomparable
    "experiment_desc": settings.VATEX_EXPERIMENT_DESC,
    "desc_id_digits": 12,    # width of the identifier derived from the clip name
}

DATA_DIR  = ROOT / "data" / "interim" / "vatex"
WORK_DIR  = DATA_DIR / "work"   # intermediate files of a single procedure
ANNOT_DIR = ROOT / "data" / "annotations" / "vatex"

TRAINING_JSON = DATA_DIR / "vatex_training_v1.0.json"
K600_CSV      = DATA_DIR / "kinetics-600_val.csv"
K400_CSV      = DATA_DIR / "kinetics-400_train.csv"
CANDIDATES    = DATA_DIR / "vatex_dev_candidates.txt"
REPORT        = DATA_DIR / f"vatex_report_{CONFIG['split']}.csv"
DESCRIPTIONS  = WORK_DIR / f"vatex_descriptions_{CONFIG['split']}.csv"
QUERIES_JSONL = ANNOT_DIR / f"vatex_queries_{CONFIG['split']}.jsonl"

SOURCE = "vatex"
STATUS_OK = "ok"   # in the download report: downloaded and verified


def youtube_id(vid_name):
    """'G9zN5TTuGO4_000179_000189' -> 'G9zN5TTuGO4'.

    Split from the RIGHT: a YouTube identifier may itself contain '_' and '-'.
    """
    return vid_name.rsplit("_", 2)[0]


def clip_duration(vid_name):
    """Clip length in seconds read from its name (end - start)."""
    _, start, end = vid_name.rsplit("_", 2)
    return float(int(end) - int(start))


def desc_id_for(vid_name):
    """Query identifier derived from the clip name.

    Same rule as src.utils.vatex.desc_id_for, repeated here rather than
    imported, because that module pulls in pandas. Numbering by position would
    move every identifier the moment one clip left the set; a digest does not.
    """
    digest = hashlib.blake2b(vid_name.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big") % 10 ** CONFIG["desc_id_digits"]


def load_kinetics(path):
    """Kinetics listing; both files carry the columns 'youtube_id' and 'label'."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def load_report():
    """(header, rows) of the download report, or (None, []) when it is absent."""
    if not REPORT.exists():
        return None, []
    with open(REPORT, encoding="utf-8-sig", newline="") as f:
        rows = [r for r in csv.reader(f, delimiter=";") if r]
    if not rows or rows[0][0] != "videoID":
        raise ValueError("Report without a header - reading by column names is impossible.")
    return rows[0], rows[1:]


def pl(x, places=1):
    """Number with a decimal comma."""
    return f"{x:.{places}f}".replace(".", ",")

## Skrypty, które trzeba uruchomić wcześniej

Wszystko poniżej uruchamia się z wiersza poleceń, z korzenia repozytorium; notatnik czyta tylko to, co te polecenia zostawią.

| # | Co | Gdzie | Kontrola |
|---|---|---|---|
| 1 | `python scripts/prepare_data/vatex_candidates.py --split dev` | wiersz poleceń | blok 2, liczebność i zmierzony udział klas |
| 2 | commit pliku `data/interim/vatex/vatex_dev_candidates.txt` | git | lista musi być w historii, zanim ruszy pobieranie |
| 3 | `python scripts/prepare_data/vatex_download.py --split dev --mode download` | wiersz poleceń, kilka godzin | blok 3, statusy pobrania |
| 4 | `python scripts/prepare_data/vatex_retry_errors.py --split dev` | wiersz poleceń | podgląd; po `--execute` wraca się do kroku 3 |
| 5 | `python scripts/prepare_data/vatex_leak_filter.py --split dev` | wiersz poleceń | powstaje `vatex_split_dev.csv` |
| 6 | blok 4, wykluczenie klipów za krótkich | notatnik | powstaje `work/clips_too_short_dev.csv` |
| 7 | blok 5, plik zapytań | notatnik | powstaje `vatex_queries_dev.jsonl` |

**Krok 1 losuje raz.** Skrypt odmawia nadpisania istniejącej listy; `--force` wolno użyć wyłącznie wtedy, gdy z poprzedniej listy nie pobrano jeszcze ani jednego klipu.

**Krok 2 jest częścią procedury, nie porządkami.** Lista zacommitowana przed pobieraniem dowodzi, że skład części deweloperskiej nie zależy od tego, co akurat dało się pobrać z YouTube.

**Zakres przebiegu bierze się z zamrożonej listy.** Przy `--split dev` skrypt sam czyta `data/interim/vatex/vatex_dev_candidates.txt` i przerabia wyłącznie wypisane tam identyfikatory. Gdy tego pliku nie ma, przerywa i odsyła do kroku 1, zamiast po cichu wziąć cały split treningowy, czyli blisko 26 tysięcy klipów.

**`--mode`** ma trzy wartości. `download` pobiera właściwe dziesięciosekundowe wycinki i mierzy każdy z nich przez ffprobe, a brakujący eksport opisów części dorabia na starcie. `check` sprawdza samą dostępność klipów na YouTube, nie pobierając materiału. `descriptions` czyta wyłącznie plik jsona i zapisuje opisy do CSV; działa lokalnie, bez ruchu sieciowego.

**Pobieranie jest wznawialne:** skrypt pomija każdy identyfikator, który jest już w raporcie, więc przebieg wolno przerwać `Ctrl+C` i uruchomić później jeszcze raz. Przy domyślnej pauzie 12 s (limit dla niezalogowanego) tysiąc klipów to kilka godzin.

**Krok 4 zwalnia wiersze nieudanych pobrań.** Wznawialność ma cenę: skrypt pobierający pomija każdy identyfikator obecny w raporcie niezależnie od statusu, więc klip zakończony błędem sam się nie ponowi i najpierw trzeba zwolnić jego wiersz. `vatex_retry_errors.py --split dev` bez `--execute` jest podglądem i niczego nie zmienia; z `--execute` zakłada kopię zapasową raportu z datą i dopiero wtedy zapisuje. Potem powtarza się krok 3. Flaga `--split` przełącza raport i katalog klipów razem, więc ponowienie części deweloperskiej nie sięga do materiału testowego.

Ponawiane są statusy przejściowe: `error`, `download_error`, `bot`, `rate_limited`, `no_format`, `login_required`. Trwałe zostają: `missing`, `private`, `geo_blocked`, `members_only`, `corrupt`. `age_restricted` wchodzi tylko z `--include-age-restricted` i tylko razem z ciasteczkami. `--check-files` dorzuca wiersze `ok`, którym brakuje pliku albo plik ma poniżej 10 kB.

**Ponawianie nie może dobierać klipów.** Operuje na tej samej zamrożonej liście i tylko dlatego jest w porządku. Czego po kilku podejściach nie da się pobrać, to przepada: dolosowanie klipów w miejsce brakujących uzależniłoby skład próby od tego, co akurat było dostępne na YouTube. Jeśli zostanie 780 zamiast 800 klipów, to jest poprawny wynik, a nie brak do uzupełnienia.

## 1. Kontrola wejścia i liczebności

Pula, z której losuje `vatex_candidates.py`: ile klipów ma split treningowy VATEX, ile z nich ma klasę Kinetics-600 mieszczącą się w słowniku Kinetics-400 i ile z nich przecieka do zbioru treningowego tego słownika. Cztery grupy to iloczyn obu podziałów.

**Przeciek liczy się przed losowaniem, a nie po nim.** Gdyby odsiewać go dopiero z wylosowanej próbki, jej liczebność i skład zależałyby od tego, ile przecieku akurat w niej wypadło, czyli od przypadku, a nie od konstrukcji. Skrypt losuje z puli już czystej i to samo liczy ten blok.

In [ ]:
k600 = {r["youtube_id"]: r["label"] for r in load_kinetics(K600_CSV)}
k400_rows = load_kinetics(K400_CSV)
k400_ids = {r["youtube_id"] for r in k400_rows}
k400_vocab = {r["label"] for r in k400_rows}

with open(TRAINING_JSON, encoding="utf-8") as f:
    training = json.load(f)

without_class = [r["videoID"] for r in training
                 if youtube_id(r["videoID"]) not in k600]

groups = {}
for record in training:
    vid = record["videoID"]
    ytid = youtube_id(vid)
    groups.setdefault((k600.get(ytid) in k400_vocab, ytid in k400_ids), []).append(vid)

total = len(training)
print(f"VATEX training split: {total} clips")
print(f"without a K600 class: {len(without_class)}   (should be 0)\n")

for in_vocab in (True, False):
    for leak in (False, True):
        count = len(groups.get((in_vocab, leak), []))
        label = "in K400 " if in_vocab else "outside "
        print(f"  class {label}| leak {'yes' if leak else 'no '}: "
              f"{count:>6}  ({pl(100 * count / total, 2):>5}%)")

clean = len(groups.get((True, False), [])) + len(groups.get((False, False), []))
print(f"\nno leak: {clean}  ({pl(100 * clean / total)}%) - this is the pool of the draw")

## 2. Stan zamrożonej listy

Liczebność wylosowanej listy i zmierzony udział klipów, których klasa Kinetics-600 mieści się w słowniku Kinetics-400.

**Losowanie jest jednostajne, bez warstwowania.** Proporcja klas ze słownika i spoza niego jest własnością części testowej, a nie wielkością zadaną z góry. Część deweloperska powstaje przed testową, więc narzucenie jej proporcji części testowej znaczyłoby, że zbiór testowy kształtuje konstrukcję zbioru deweloperskiego. Udział jest tu więc mierzony, a nie zadany, i tak samo raportowany.

In [ ]:
if not CANDIDATES.exists():
    print(f"No {CANDIDATES.name} yet - run step 1 of the table above.")
else:
    k600 = {r["youtube_id"]: r["label"] for r in load_kinetics(K600_CSV)}
    k400_vocab = {r["label"] for r in load_kinetics(K400_CSV)}

    drawn = CANDIDATES.read_text(encoding="utf-8").split()
    in_vocab = [v for v in drawn if k600.get(youtube_id(v)) in k400_vocab]
    outside = [v for v in drawn if k600.get(youtube_id(v)) not in k400_vocab]

    print(f"frozen list:   {len(drawn):>5}  (drawn {CONFIG['sample_size']}, "
          f"seed {CONFIG['seed']}, uniform over the clean pool)")
    print(f"unique:        {len(set(drawn)):>5}")
    print(f"class in K400: {len(in_vocab):>5}  "
          f"({pl(100 * len(in_vocab) / len(drawn)):>5}%)   measured, not imposed")
    print(f"class outside: {len(outside):>5}  "
          f"({pl(100 * len(outside) / len(drawn)):>5}%)")

## 3. Co się pobrało

Statusy z rejestru pobrania: ile klipów ma status `ok`, ile przepadło i z jakiego powodu, ile zostało jeszcze do przerobienia. Gdy pliku jeszcze nie ma, blok mówi, że pobieranie nie ruszyło; to nie jest błąd, tylko kolejność kroków.

In [ ]:
header, rows = load_report()

if header is None:
    print(f"No {REPORT.name} - the download has not started yet.")
    print("This is not an error: it is step 4 of the table above.")
else:
    i_status = header.index("status")
    statuses = Counter(r[i_status] for r in rows)
    processed = {r[0] for r in rows}
    drawn = (CANDIDATES.read_text(encoding="utf-8").split()
             if CANDIDATES.exists() else [])
    n = len(rows)

    print(f"drawn:       {len(drawn):>5}")
    print(f"processed:   {n:>5}")
    print(f"downloaded:  {statuses.get(STATUS_OK, 0):>5}  "
          f"({pl(100 * statuses.get(STATUS_OK, 0) / n)}% of processed)")
    print(f"lost:        {n - statuses.get(STATUS_OK, 0):>5}")
    print(f"remaining:   {len(set(drawn) - processed):>5}\n")

    for status, count in statuses.most_common():
        print(f"  {status:<18} {count:>5}  {pl(100 * count / n):>5}%")

## 4. Klipy za krótkie

Próg jest jednostronny. Klip dłuższy od nominalnego jest nieszkodliwy - ffmpeg dodał margines przy cięciu do klatki kluczowej, a opisywane zdarzenie nadal jest w środku. Klip krótszy oznacza brakujący materiał: nagranie źródłowe skończyło się, zanim zamknęło się okno adnotacji. Próg jest ten sam co dla części testowej (`settings.VATEX_MIN_CLIP_S`), bo część deweloperska filtrowana inaczej niż testowa nie byłaby z nią porównywalna.

**Tu nie ma przeglądu ręcznego.** Część testowa ogląda te klipy pojedynczo i odrzucone przenosi do `clips/rejected/`, bo każdy z nich wchodzi do kolekcji ocenianej w pracy. Część deweloperska służy wyłącznie wyznaczeniu progu dopasowania i kontroli sygnału ruchu, więc klipy wykluczane są automatycznie, bez oglądania i bez katalogu kwarantanny: klip krótszy od okna adnotacji nie niesie opisywanego zdarzenia niezależnie od tego, co na nim widać.

**Zapisuje:** `data/interim/vatex/work/clips_too_short_dev.csv` - identyfikator i długość każdego wykluczonego klipu, od najkrótszego. Blok nadaje im w raporcie status `too_short`, więc wypadają z grupy `ok` i nie wchodzą do pliku zapytań. Status jest trwały: ponowne pobranie nie wydłuży nagrania źródłowego, więc `vatex_retry_errors.py` go nie rusza.

Uruchamiaj po zakończeniu pobierania - blok przepisuje raport, który pobieranie w tym czasie zapisuje. Można go powtarzać: klipy już oznaczone zostają w pliku i nie są liczone drugi raz.

In [ ]:
STATUS_TOO_SHORT = "too_short"
TOO_SHORT_CSV = WORK_DIR / f"clips_too_short_{CONFIG['split']}.csv"
THRESHOLD = settings.VATEX_MIN_CLIP_S

header, rows = load_report()

if header is None:
    print(f"No {REPORT.name} - the download has not started yet.")
else:
    i_status, i_duration = header.index("status"), header.index("duration")

    def duration_of(row):
        """Measured duration, or None when the report has no number for it."""
        raw = row[i_duration].strip().replace(",", ".") if len(row) > i_duration else ""
        return float(raw) if raw else None

    # Rows already marked count too: the cell has to be repeatable, and without
    # them a second run would rewrite the file with an empty one.
    short = {}
    for row in rows:
        if row[i_status] not in (STATUS_OK, STATUS_TOO_SHORT):
            continue
        seconds = duration_of(row)
        if seconds is not None and seconds < THRESHOLD:
            short[row[0]] = seconds

    fresh = sum(1 for r in rows if r[0] in short and r[i_status] == STATUS_OK)
    ok_before = sum(1 for r in rows if r[i_status] == STATUS_OK)
    missing_duration = sum(1 for r in rows
                           if r[i_status] == STATUS_OK and duration_of(r) is None)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with open(TOO_SHORT_CSV, "w", encoding="utf-8", newline="") as f:
        f.write("videoID;duration\n")
        for vid, seconds in sorted(short.items(), key=lambda pair: pair[1]):
            f.write(f"{vid};{seconds:.2f}\n")   # decimal point: this is a data file

    for row in rows:
        if row[0] in short:
            row[i_status] = STATUS_TOO_SHORT
    with open(REPORT, "w", encoding="utf-8-sig", newline="") as f:
        csv.writer(f, delimiter=";").writerows([header] + rows)

    print(f"clips with status '{STATUS_OK}' before: {ok_before:>5}")
    print(f"shorter than {pl(THRESHOLD)} s:            {len(short):>5}  "
          f"(newly excluded now: {fresh})")
    print(f"remaining in the part:            {ok_before - fresh:>5}")
    if missing_duration:
        print(f"WARNING: {missing_duration} clips have status 'ok' but no measured "
              "duration - they were not checked")
    print(f"saved {len(short)} entries to {TOO_SHORT_CSV.name}")

## 5. Plik zapytań części deweloperskiej

**Zapisuje:** `data/annotations/vatex/vatex_queries_dev.jsonl` - po jednym rekordzie na pobrany klip (status `ok`), w schemacie wspólnym dla wszystkich trzech zbiorów (`src/utils/queries.py`).

Zapytaniem jest pierwszy opis angielski klipu, ta sama konwencja co w części testowej. `vid_name` wskazuje klip, `ts` to jego granice na własnej osi czasu (klip jest już wycinkiem, więc zaczyna się od zera), a `event_id` to identyfikator klipu, bo poprawną odpowiedzią jest ten właśnie klip. Przynależność do części niesie nazwa pliku, tak samo jak w serialach; rekord nie ma osobnego pola.

`requirements`, `complexity` i `identities` zostają puste i takie zostaną: część deweloperska służy wyłącznie wyznaczeniu progu dopasowania fraz i kontroli sygnału ruchu.

Blok czyta rejestr i opisy z dysku, nie z poprzednich bloków, więc po zakończeniu pobierania można uruchomić sam ten.

In [ ]:
header, rows = load_report()

if header is None:
    print(f"No {REPORT.name} - nothing to build queries from yet.")
elif not DESCRIPTIONS.exists():
    print(f"No {DESCRIPTIONS.name} - run mode 'descriptions' first (step 3).")
else:
    i_status = header.index("status")
    ok_vids = sorted({r[0] for r in rows if r[i_status] == STATUS_OK})

    first = {}
    with open(DESCRIPTIONS, encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f, delimiter=";"):
            if int(row["desc_no"]) == CONFIG["experiment_desc"]:
                first[row["videoID"]] = str(row["desc_en"]).strip()

    without_desc = [vid for vid in ok_vids if vid not in first]
    records = [Query(desc_id=desc_id_for(vid),
                     desc=first[vid],
                     vid_name=vid,
                     ts=(0.0, clip_duration(vid)),
                     source=SOURCE,
                     event_id=vid)
               for vid in ok_vids if vid in first]

    repeated = {i for i, c in Counter(q.desc_id for q in records).items() if c > 1}
    if repeated:
        raise ValueError(f"derived desc_id collides for {len(repeated)} clips "
                         "- widen CONFIG['desc_id_digits']")

    written = save_jsonl(records, QUERIES_JSONL)

    print(f"clips with status '{STATUS_OK}': {len(ok_vids)}")
    if without_desc:
        print(f"WARNING: {len(without_desc)} of them have no description "
              f"(e.g. {without_desc[0]}) - re-run mode 'descriptions'")
    print(f"queries: {len(records)}")
    print("requirements/complexity/identities: empty by construction")
    print(f"saved -> {written.relative_to(ROOT)}")